# Task 1: Vectorized Scaled Dot-Product Attention from Scratch

## Mathematical Formula
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}} + M\right) V$$


In [ ]:
import numpy as np

# Core scaled dot-product attention calculation handling 4D tensors and causal masking
def scaled_dot_product_attention(Q: np.ndarray, K: np.ndarray, V: np.ndarray, mask: np.ndarray = None):
    d_k = Q.shape[-1]
    scores = np.matmul(Q, K.swapaxes(-2, -1)) / np.sqrt(d_k)
    
    if mask is not None:
        scores = np.where(mask == 1, -1e9, scores)
        
    exp_scores = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    attention_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
    output = np.matmul(attention_weights, V)
    return output, attention_weights


In [ ]:
# Initialize dummy multi-head 4D tensors and verify upper-triangular causal masking
batch_size, num_heads, seq_len, head_dim = 2, 4, 6, 16
np.random.seed(42)
Q = np.random.randn(batch_size, num_heads, seq_len, head_dim)
K = np.random.randn(batch_size, num_heads, seq_len, head_dim)
V = np.random.randn(batch_size, num_heads, seq_len, head_dim)

causal_mask = np.triu(np.ones((seq_len, seq_len)), k=1)
output, attn_weights = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

print("Output shape:", output.shape)
print("Attention weights shape:", attn_weights.shape)
assert np.allclose(np.triu(attn_weights[0, 0], k=1), 0.0)
print("[SUCCESS] Causal attention masking verified.")
